# Deskripsi Tugas
"Sistem ini dirancang untuk melakukan penghitungan berbasis puncak (peak count). Untuk setiap video, sistem memproses seluruh frame dan melaporkan jumlah maksimum orang yang terdeteksi secara bersamaan dalam satu frame sebagai hasil prediksi akhir. Metodologi ini dipilih untuk mengukur kepadatan maksimum orang dalam sebuah adegan dan merupakan pendekatan standar untuk validasi awal sistem penghitungan."

# LANGKAH 0: SETUP LINGKUNGAN

In [3]:
# =============================================================
# NOTEBOOK VALIDASI AKURASI SISTEM HUMAN COUNTING
# =============================================================
# Tujuan: Memvalidasi performa model fine-tuned pada tugas
#         menghitung orang dalam video dan membuat laporan
#         akurasi hitungan.
# =============================================================

# -----------------------------------------------------
# LANGKAH 0: SETUP LINGKUNGAN
# -----------------------------------------------------
print("📦 Menginstal/Memeriksa library yang dibutuhkan...")
!pip install ultralytics pandas openpyxl -q

import os
import cv2
import pandas as pd
from ultralytics import YOLO
from google.colab import drive
from google.colab.patches import cv2_imshow
import time

print("✅ Library berhasil diimpor.")

# Hubungkan ke Google Drive
drive.mount('/content/drive')
print("✅ Google Drive berhasil terhubung.")

📦 Menginstal/Memeriksa library yang dibutuhkan...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 120.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 43.3 MB/s eta 0:0

In [4]:
# -----------------------------------------------------
# LANGKAH 1: KONFIGURASI & PEMUATAN MODEL
# -----------------------------------------------------
print("\n⚙️ Memuat konfigurasi dan model...")

# === KONFIGURASI WAJIB DIISI ===
# 1. Path ke model terbaik Anda (best.pt dari training Anda)
PATH_MODEL_TERBAIK = '/content/drive/MyDrive/Magang KP Model AI/Model Total Count/v0.5.0/PersonDetection_LargeDataset_Run1/weights/best.pt'

# 2. Path ke folder di Google Drive tempat Anda menyimpan 10 video tes
PATH_FOLDER_VIDEO_INPUT = '/content/drive/MyDrive/Magang KP Model AI/Model Total Count/Video Uji Human Count/'

# 3. Path ke folder di Google Drive untuk menyimpan HASIL (video + laporan)
PATH_FOLDER_OUTPUT = '/content/drive/MyDrive/Magang KP Model AI/Model Total Count/Hasil Validasi Human Count/'
os.makedirs(PATH_FOLDER_OUTPUT, exist_ok=True) # Buat folder jika belum ada

# 4. KUNCI JAWABAN (GROUND TRUTH) - (Wajib Diisi Manual!)
# Format: 'nama_file_video.mp4': jumlah_orang_sebenarnya
# PENTING: Hitung manual dan isi jumlah orang terbanyak di setiap video.
GROUND_TRUTH_COUNTS = {
    'video_keramaian_1.mp4': 2,
    'video_keramaian_2.mp4': 2,
    'video_keramaian_3.mp4': 4,
    'video_keramaian_4.mp4': 5,
    'video_keramaian_5.mp4': 3,
    # Lanjutkan sampai 10 video...
    # 'nama_video_6.mp4': 6,
    # 'nama_video_7.mp4': 1,
    # 'nama_video_8.mp4': 10,
    # 'nama_video_9.mp4': 7,
    # 'nama_video_10.mp4': 9,
}
# ===============================

try:
    model = YOLO(PATH_MODEL_TERBAIK)
    # Mencari tahu ID kelas 'person'. Kita asumsikan dari notebook Anda,
    # urutannya: gun, person, person with a gun. Jadi, 'person' adalah ID 1.
    # Kita cek untuk memastikan.
    person_class_id = -1
    for class_id, class_name in model.names.items():
        if class_name.lower() == 'person':
            person_class_id = class_id
            break

    if person_class_id == -1:
         raise ValueError("Kelas 'person' tidak ditemukan di model.names. Cek file data.yaml Anda.")

    print(f"✅ Model berhasil dimuat dari: {PATH_MODEL_TERBAIK}")
    print(f"✅ Kelas 'person' teridentifikasi dengan ID: {person_class_id}")
    print(f"✅ Siap memproses video dari: {PATH_FOLDER_VIDEO_INPUT}")
    print(f"✅ Hasil akan disimpan di: {PATH_FOLDER_OUTPUT}")
except Exception as e:
    print(f"❌ Gagal memuat model atau konfigurasi: {e}")
    # Berhenti jika model gagal dimuat
    exit()




⚙️ Memuat konfigurasi dan model...
✅ Model berhasil dimuat dari: /content/drive/MyDrive/Magang KP Model AI/Model Total Count/v0.5.0/PersonDetection_LargeDataset_Run1/weights/best.pt
✅ Kelas 'person' teridentifikasi dengan ID: 1
✅ Siap memproses video dari: /content/drive/MyDrive/Magang KP Model AI/Model Total Count/Video Uji Human Count/
✅ Hasil akan disimpan di: /content/drive/MyDrive/Magang KP Model AI/Model Total Count/Hasil Validasi Human Count/


In [11]:
# ================================================================
# LANGKAH 2: FUNGSI UTAMA UNTUK PROSES & VALIDASI VIDEO (PERBAIKAN FINAL)
# ================================================================
# Logika Skor Persentase diperbaiki untuk menangani over-prediction dengan benar.
# ================================================================

def validate_human_count_on_videos(video_folder, output_folder, model, ground_truth):
    """
    Fungsi untuk memproses semua video di folder, menghitung orang,
    membandingkan dengan ground truth menggunakan berbagai metrik,
    dan menyimpan hasilnya.
    """
    video_files = [f for f in os.listdir(video_folder) if f.endswith(('.mp4', '.avi', '.mov'))]
    report_data = []

    # Variabel untuk setiap metrik
    correctly_predicted_videos = 0 # Untuk Exact Match
    total_absolute_error = 0       # Untuk MAE
    total_percentage_score = 0     # Untuk Metode Skor Baru yang Adil

    print("\n" + "="*50)
    print("🚀 MEMULAI PROSES VALIDASI PADA SEMUA VIDEO...")
    print("="*50)

    for video_name in video_files:
        if video_name not in ground_truth:
            print(f"\n⚠️ Peringatan: Video '{video_name}' ada di folder tapi tidak ada di GROUND_TRUTH_COUNTS. Video ini akan dilewati.")
            continue

        video_path = os.path.join(video_folder, video_name)
        output_video_path = os.path.join(output_folder, f"hasil_{video_name}")

        print(f"\n🔄 Memproses: {video_name}")
        start_time = time.time()

        cap = cv2.VideoCapture(video_path)
        # ... (sisa kode pembukaan video dan video writer sama persis) ...
        frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = int(cap.get(cv2.CAP_PROP_FPS))
        out = cv2.VideoWriter(output_video_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame_width, frame_height))
        frame_by_frame_counts = []

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            # --- Tuning Point: Ubah `conf` di sini jika perlu ---
            detections = model.predict(frame, classes=[person_class_id], conf=0.7, verbose=False)
            current_person_count = len(detections[0].boxes)
            frame_by_frame_counts.append(current_person_count)

            # ... (kode menggambar anotasi ke frame sama persis) ...
            annotated_frame = detections[0].plot()
            cv2.putText(annotated_frame, f"Jumlah Terdeteksi: {current_person_count}",
                        (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 3, (0, 255, 255), 4)
            if frame_by_frame_counts:
                 cv2.putText(annotated_frame, f"Prediksi Final (Max Sejauh Ini): {max(frame_by_frame_counts)}",
                             (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 3, (0, 255, 0), 4)

            out.write(annotated_frame)

        cap.release()
        out.release()

        max_detected_count_in_video = 0
        if frame_by_frame_counts:
            max_detected_count_in_video = max(frame_by_frame_counts)

        # === PERHITUNGAN SEMUA METRIK DENGAN LOGIKA YANG BENAR ===
        actual_count = ground_truth.get(video_name)
        is_correct = False
        current_error = 0
        percentage_score_for_video = 0.0

        if actual_count is not None:
            # Hitung Error
            current_error = abs(actual_count - max_detected_count_in_video)
            total_absolute_error += current_error

            # Kalkulasi Skor Persentase yang adil (1 - Error Relatif)
            if actual_count > 0:
                relative_error = current_error / actual_count
                percentage_score_for_video = 1.0 - relative_error
                # Pastikan skor tidak negatif jika error sangat besar
                percentage_score_for_video = max(0.0, percentage_score_for_video)
            elif current_error == 0: # Kasus di mana actual_count adalah 0 dan prediksi juga 0
                 percentage_score_for_video = 1.0

            total_percentage_score += percentage_score_for_video

            # Kalkulasi Exact Match
            is_correct = (current_error == 0)
            if is_correct:
                correctly_predicted_videos += 1

        end_time = time.time()
        duration = end_time - start_time

        print(f"   - Waktu Proses: {duration:.2f} detik")
        print(f"   - Jumlah Sebenarnya (Manual): {actual_count}")
        print(f"   - Prediksi Sistem (Max Count):  {max_detected_count_in_video}")
        print(f"   - Skor Persentase (Revisi): {percentage_score_for_video:.2%}") # Diperbaiki
        print(f"   - Hasil Exact Match: {'✅ BENAR' if is_correct else '❌ SALAH'}")

        report_data.append({
            "Nama Video": video_name,
            "Jml Sebenarnya (GT)": actual_count,
            "Jml Prediksi Sistem": max_detected_count_in_video,
            "Skor Persentase": f"{percentage_score_for_video:.2%}", # Diperbaiki
            "Hasil Exact Match": "Benar" if is_correct else "Salah",
            "Error (Selisih)": current_error
        })

    num_videos_processed = len(report_data)
    if num_videos_processed > 0:
        final_percentage_accuracy = (total_percentage_score / num_videos_processed) * 100
        final_exact_accuracy = (correctly_predicted_videos / num_videos_processed) * 100
        mean_absolute_error = (total_absolute_error / num_videos_processed)
    else:
        final_percentage_accuracy = 0; final_exact_accuracy = 0; mean_absolute_error = 0

    return report_data, final_percentage_accuracy, final_exact_accuracy, mean_absolute_error


In [12]:

# =====================================================
# LANGKAH 3: EKSEKUSI & PELAPORAN (PERBAIKAN FINAL)
# =====================================================
# Panggilan fungsi sama, tapi judul di laporan diubah agar lebih jelas
report, main_accuracy, exact_accuracy, mae = validate_human_count_on_videos(
    PATH_FOLDER_VIDEO_INPUT,
    PATH_FOLDER_OUTPUT,
    model,
    GROUND_TRUTH_COUNTS
)

# Tampilkan Laporan Akhir yang Komprehensif
print("\n" + "="*60)
print("✨ LAPORAN AKHIR VALIDASI SISTEM HUMAN COUNTING ✨")
print("="*60)
print(f"🎯 AKURASI UTAMA (Berdasarkan Skor Persentase): {main_accuracy:.2f}%")
print(f"   (Skor dihitung sbg 1 - |Error|/Aktual. Lebih adil utk over-prediction)")
print("-" * 30)
print("METRIK TAMBAHAN UNTUK ANALISIS LEBIH DALAM:")
print(f"📊 Akurasi Ketepatan (Exact Match): {exact_accuracy:.2f}%")
print(f"   (Persentase video yang hitungannya 100% benar)")
print("")
print(f"📈 Mean Absolute Error (MAE): {mae:.2f}")
print(f"   (Rata-rata, sistem meleset sebanyak {mae:.2f} orang per video)")
print("-" * 60)

# Rangkuman Detail
df_report = pd.DataFrame(report)
kolom_urut = [
    "Nama Video",
    "Jml Sebenarnya (GT)",
    "Jml Prediksi Sistem",
    "Skor Persentase",
    "Hasil Exact Match",
    "Error (Selisih)"
]
print("📋 Rangkuman Hasil Detail per Video:")
print(df_report[kolom_urut].to_string(index=False))

# Simpan ke Excel
excel_report_path = os.path.join(PATH_FOLDER_OUTPUT, 'Laporan_Akurasi_Final_Human_Count.xlsx')
try:
    df_report[kolom_urut].to_excel(excel_report_path, index=False, engine='openpyxl')
    print("-" * 60)
    print(f"✅ Laporan lengkap berhasil disimpan sebagai file Excel di:")
    print(f"   {excel_report_path}")
except Exception as e:
     print(f"❌ Gagal menyimpan ke Excel. Error: {e}")
print("==========================================================")


🚀 MEMULAI PROSES VALIDASI PADA SEMUA VIDEO...

🔄 Memproses: video_keramaian_1.mp4
   - Waktu Proses: 14.81 detik
   - Jumlah Sebenarnya (Manual): 2
   - Prediksi Sistem (Max Count):  2
   - Skor Persentase (Revisi): 100.00%
   - Hasil Exact Match: ✅ BENAR

🔄 Memproses: video_keramaian_2.mp4
   - Waktu Proses: 53.68 detik
   - Jumlah Sebenarnya (Manual): 2
   - Prediksi Sistem (Max Count):  2
   - Skor Persentase (Revisi): 100.00%
   - Hasil Exact Match: ✅ BENAR

🔄 Memproses: video_keramaian_3.mp4
   - Waktu Proses: 37.22 detik
   - Jumlah Sebenarnya (Manual): 4
   - Prediksi Sistem (Max Count):  4
   - Skor Persentase (Revisi): 100.00%
   - Hasil Exact Match: ✅ BENAR

🔄 Memproses: video_keramaian_4.mp4
   - Waktu Proses: 46.42 detik
   - Jumlah Sebenarnya (Manual): 5
   - Prediksi Sistem (Max Count):  5
   - Skor Persentase (Revisi): 100.00%
   - Hasil Exact Match: ✅ BENAR

🔄 Memproses: video_keramaian_5.mp4
   - Waktu Proses: 40.12 detik
   - Jumlah Sebenarnya (Manual): 3
   - Predik